# 🧩 02 — Tools, Specialist Agents and the Supervisor Graph

Notebook 01 built an agent that answers from documents. This one builds the agent that
*changes things*: looks up invoices, issues refunds, changes plans, opens tickets. Actions
are where evaluation gets hard, because a wrong answer is embarrassing and a wrong action
costs money.

## Learning Objectives
In this notebook, you will learn:
1. **SQLite as ground truth** — why the final state of a database is the only end-to-end
   check no framework can fake
2. **Tools with an approval gate** — writes that pause for a human, and the audit trail that
   makes "did it wait?" a testable question
3. **Supervisor routing and handoffs** — one classifier, four specialists, and the context
   that has to survive the hop between them
4. **The trajectory dict** — one run, one record; the single data structure every level-2
   and level-3 metric in this course consumes
5. **Record-and-replay fixtures** — so notebooks 03–12 run deterministically, with no API key

## Prerequisites
- Notebook 01 completed (`data/chroma/` exists)
- `OPENAI_API_KEY` only if you set `FIXTURE_MODE = "record"`; replay needs no key

---
## 🗺️ The concept

```
                       Customer message
                             |
                      +------v------+
                      |  Supervisor |   classifies intent, routes
                      +------+------+
      +----------------+-----+-----------+-----------------+
  +---v------+  +------v----+  +---------v-+  +------------v-+
  |Knowledge |  |  Billing  |  |  Account  |  |  Escalation  |
  |agent(RAG)|  |   agent   |  |   agent   |  |    agent     |
  +---+------+  +------+----+  +--------+--+  +-----------+--+
      |                |                |                 |
  help-center     get_invoice      get_account       create_ticket
  docs (nb 01)    issue_refund*    change_plan*
                                   reset_password
                  * writes pause for human approval
                             |
                  SQLite = the final state
    (accounts, invoices, refunds, tickets, approvals)
```

The canonical run this course returns to over and over:

> *"I was double charged, and what's your refund policy?"*

Supervisor → Knowledge agent (cited policy) → Billing agent (invoice lookup, refund request)
→ approval → reply. One sentence that exercises RAG, tool calls, a handoff, an approval gate
and a database write. Every one of those is a separate place to fail, and notebook 06's blame
attribution exists to tell them apart.

### Key Concepts:
- **Three levels of agentic evaluation.** Tool call → agent/handoff → end-to-end run. A
  failure at level 3 is meaningless until you trace it to the level that caused it.
- **`result_status` is not the tool's return value.** `ok` / `empty` / `unavailable` are
  three different situations that deserve three different agent behaviours, and conflating
  them is one of the most common real bugs in production agents.
- **The database is the referee.** An LLM judge can be argued with. `SELECT SUM(amount) FROM
  refunds WHERE invoice_id = ?` cannot.

---
## ⚙️ Part 0: Setup and fault switches

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: paths, pins, imports
# ============================================================================
import os
import json
import re
import time
import hashlib
import sqlite3
import warnings
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
FIXTURE_DIR = os.path.join(DATA_DIR, "fixtures")
DB_PATH = os.path.join(DATA_DIR, "lumen.db")
os.makedirs(FIXTURE_DIR, exist_ok=True)

print("✅ Setup complete")
print("🗄️  database ->", DB_PATH)
print("🎬 fixtures ->", FIXTURE_DIR)

In [ ]:
# ============================================================================
# FAULT SWITCHES: the eight ways this system is allowed to break
# ============================================================================
FAULTS = {
    # --- tool layer -------------------------------------------------------
    "wrong_tool": False,           # call get_account when get_invoice was needed
    "bad_arguments": False,        # right tool, wrong customer id
    "tool_returns_empty": False,   # tool succeeds but finds nothing

    # --- agent / handoff layer -------------------------------------------
    "wrong_route": False,          # supervisor sends billing questions to account
    "lost_handoff_context": False, # the second agent forgets what the first found

    # --- safety -----------------------------------------------------------
    "skip_approval": False,        # write to the DB without waiting for a human

    # --- cost and efficiency ----------------------------------------------
    "retry_loop": False,           # call the same tool three times
    "verbose_prompt": False,       # pad the system prompt, watch cost climb
}

print("🎛️  Faults armed:", {k: v for k, v in FAULTS.items() if v} or "none (clean run)")

---
## 🗄️ Part 1: SQLite — the ground truth

Five tables. The one that matters most is `approvals`, because it is what turns "the agent
should wait for a human" from a policy into a **row that either exists or doesn't**.

Note the seed data: invoice `INV-2041` is charged twice. That duplicate is the spine of the
canonical run, and notebook 06 checks that exactly one refund of exactly the right amount
lands against it — not zero, and importantly not two.

In [ ]:
# ============================================================================
# SCHEMA + SEED: rebuild the database from scratch, every run
# ============================================================================
# Idempotent by construction. An evaluation suite that depends on leftover state
# from the last run is an evaluation suite that lies to you eventually.

def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


SCHEMA = """
DROP TABLE IF EXISTS accounts;
DROP TABLE IF EXISTS invoices;
DROP TABLE IF EXISTS refunds;
DROP TABLE IF EXISTS tickets;
DROP TABLE IF EXISTS approvals;

CREATE TABLE accounts (
    customer_id TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT NOT NULL,
    plan        TEXT NOT NULL,
    region      TEXT NOT NULL,
    lang        TEXT NOT NULL
);

CREATE TABLE invoices (
    invoice_id  TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    amount      REAL NOT NULL,
    currency    TEXT NOT NULL,
    issued_on   TEXT NOT NULL,
    status      TEXT NOT NULL      -- paid | open | disputed
);

CREATE TABLE refunds (
    refund_id   TEXT PRIMARY KEY,
    invoice_id  TEXT NOT NULL,
    customer_id TEXT NOT NULL,
    amount      REAL NOT NULL,
    approved_by TEXT,              -- NULL means it was never approved
    created_at  TEXT NOT NULL
);

CREATE TABLE tickets (
    ticket_id   TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    summary     TEXT NOT NULL,
    priority    TEXT NOT NULL,
    created_at  TEXT NOT NULL
);

CREATE TABLE approvals (
    approval_id TEXT PRIMARY KEY,
    action      TEXT NOT NULL,     -- issue_refund | change_plan
    payload     TEXT NOT NULL,     -- JSON of the proposed arguments
    decision    TEXT,              -- approved | rejected | NULL while pending
    decided_by  TEXT,
    requested_at TEXT NOT NULL,
    decided_at   TEXT
);
"""

ACCOUNTS = [
    ("CUS-100", "Ana Ruiz",      "ana@example.com",   "Growth",  "eu-west", "es"),
    ("CUS-101", "Ben Okafor",    "ben@example.com",   "Starter", "us-east", "en"),
    ("CUS-102", "Priya Nair",    "priya@example.com", "Scale",   "ap-south", "hi"),
    ("CUS-103", "Dan Whitfield", "dan@example.com",   "Growth",  "us-east", "en"),
    ("CUS-104", "Elif Kaya",     "elif@example.com",  "Starter", "eu-west", "en"),
    ("CUS-105", "Marco Silva",   "marco@example.com", "Growth",  "eu-west", "es"),
]

INVOICES = [
    # CUS-101's double charge: same amount, same day, two invoice ids.
    ("INV-2041", "CUS-101",  49.00, "USD", "2026-08-01", "disputed"),
    ("INV-2042", "CUS-101",  49.00, "USD", "2026-08-01", "paid"),
    ("INV-2043", "CUS-100", 199.00, "EUR", "2026-08-03", "paid"),
    ("INV-2044", "CUS-102", 899.00, "USD", "2026-08-03", "paid"),
    ("INV-2045", "CUS-103", 199.00, "USD", "2026-07-28", "paid"),
    ("INV-2046", "CUS-104",  49.00, "EUR", "2026-06-02", "paid"),   # >30 days: not eligible
    ("INV-2047", "CUS-105", 199.00, "EUR", "2026-08-05", "open"),
]


def reset_database():
    conn = connect()
    conn.executescript(SCHEMA)
    conn.executemany("INSERT INTO accounts VALUES (?,?,?,?,?,?)", ACCOUNTS)
    conn.executemany("INSERT INTO invoices VALUES (?,?,?,?,?,?)", INVOICES)
    conn.commit()
    conn.close()


reset_database()

conn = connect()
print("✅ Database rebuilt")
for table in ["accounts", "invoices", "refunds", "tickets", "approvals"]:
    n = conn.execute("SELECT COUNT(*) FROM " + table).fetchone()[0]
    print("   %-10s %d rows" % (table, n))
conn.close()

In [ ]:
# ============================================================================
# THE DUPLICATE: the fact the canonical run is built around
# ============================================================================
conn = connect()
rows = conn.execute(
    "SELECT invoice_id, amount, issued_on, status FROM invoices WHERE customer_id = ?",
    ("CUS-101",),
).fetchall()
conn.close()

print("🧾 Invoices for CUS-101 (Ben Okafor):")
for r in rows:
    print("   %s  %6.2f  %s  %s" % (r["invoice_id"], r["amount"], r["issued_on"], r["status"]))
print()
print("⚠️  Two 49.00 charges on the same day. Correct behaviour: refund exactly ONE of them.")

---
## 🎬 Part 2: Record-and-replay

Every LLM call in this course goes through one function. In `replay` mode it reads a JSON
file keyed by a hash of the request; in `record` mode it calls the real API and writes that
file. The consequence is worth stating plainly:

**Notebooks 03–12 and the CI gate run with no API key and no cost, and give the same numbers
every time.** Evaluation results that change when nothing changed are not measurements.

> **Note**: `record` mode makes real, paid API calls. It is off by default and you have to
> set it deliberately.

In [ ]:
# ============================================================================
# FIXTURES: hash the request, replay the response
# ============================================================================
FIXTURE_MODE = "replay"      # "replay" = free and deterministic (default)
                             # "record" = LIVE API CALLS, costs money

_llm = None                  # built lazily, so replay mode never needs a key


def get_llm():
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        # from langchain_ollama import ChatOllama
        # _llm = ChatOllama(model="llama3.1", temperature=0)
    return _llm


def fixture_key(messages):
    """A stable id for a request. Sorted keys so dict ordering can't change it."""
    blob = json.dumps(messages, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def call_model(messages):
    """The ONLY place this course talks to a language model.

    Returns (text, token_estimate). Token counts are estimated from characters
    rather than tokenized: this course needs cost to be comparable between runs,
    not accurate to the token.
    """
    key = fixture_key(messages)
    path = os.path.join(FIXTURE_DIR, key + ".json")

    if FIXTURE_MODE == "replay":
        if not os.path.exists(path):
            raise FileNotFoundError(
                "No fixture for this request (" + key + ").\n"
                "Either you changed a prompt, or this request was never recorded.\n"
                "Set FIXTURE_MODE = 'record' to call the live API — this costs money."
            )
        with open(path, encoding="utf-8") as f:
            saved = json.load(f)
        return saved["output"], saved["tokens"]

    # --- record mode: live call, then persist ---
    text = get_llm().invoke(messages).content.strip()
    tokens = sum(len(m["content"]) for m in messages) // 4 + len(text) // 4
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"messages": messages, "output": text, "tokens": tokens},
                  f, ensure_ascii=False, indent=2)
    return text, tokens


n_fixtures = len([f for f in os.listdir(FIXTURE_DIR) if f.endswith(".json")])
print("🎬 FIXTURE_MODE =", FIXTURE_MODE)
print("📁 fixtures on disk:", n_fixtures)

---
## 🔧 Part 3: Tools

Six tools, split into reads and writes. The split is the point: reads are safe to retry,
writes are not, and the two write tools (`issue_refund`, `change_plan`) refuse to touch the
database unless an approved row exists in `approvals`.

### Why not LangGraph's `interrupt()`?
Production would use it. Here the approval is an explicit table plus an explicit check,
because that makes the safety property **checkable from outside the agent**: notebook 07
asserts against `refunds.approved_by IS NULL` without knowing anything about the graph. A
safety check that has to trust the framework it is checking is not much of a check.

In [ ]:
# ============================================================================
# READ TOOLS: safe, idempotent, retryable
# ============================================================================
from langchain_core.tools import tool


@tool
def get_account(customer_id: str) -> dict:
    """Look up a customer's account: plan, region, email and language."""
    conn = connect()
    row = conn.execute("SELECT * FROM accounts WHERE customer_id = ?",
                       (customer_id,)).fetchone()
    conn.close()
    if row is None:
        return {"status": "empty", "data": None}
    return {"status": "ok", "data": dict(row)}


@tool
def get_invoice(customer_id: str, invoice_id: str = "") -> dict:
    """List a customer's invoices, or fetch one invoice by id."""
    conn = connect()
    if invoice_id:
        rows = conn.execute(
            "SELECT * FROM invoices WHERE customer_id = ? AND invoice_id = ?",
            (customer_id, invoice_id)).fetchall()
    else:
        rows = conn.execute("SELECT * FROM invoices WHERE customer_id = ?",
                            (customer_id,)).fetchall()
    conn.close()

    if FAULTS["tool_returns_empty"]:
        rows = []                       # the fault: the query "works" and finds nothing

    if not rows:
        # `empty` is NOT an error. The agent has to decide what to do about it,
        # and notebook 05 scores that decision.
        return {"status": "empty", "data": []}
    return {"status": "ok", "data": [dict(r) for r in rows]}


print("✅ Read tools: get_account, get_invoice")

In [ ]:
# ============================================================================
# APPROVAL GATE: request -> human decision -> only then, the write
# ============================================================================

def request_approval(action, payload):
    """Record a pending approval and return its id. Writes nothing else."""
    approval_id = "APR-" + hashlib.sha1(
        (action + json.dumps(payload, sort_keys=True) + str(time.time())
         ).encode()).hexdigest()[:8]
    conn = connect()
    conn.execute(
        "INSERT INTO approvals VALUES (?,?,?,?,?,?,?)",
        (approval_id, action, json.dumps(payload), None, None,
         time.strftime("%Y-%m-%dT%H:%M:%S"), None),
    )
    conn.commit()
    conn.close()
    return approval_id


def decide_approval(approval_id, decision, decided_by):
    """A human approves or rejects. Notebook 09 evaluates THIS step too."""
    conn = connect()
    conn.execute(
        "UPDATE approvals SET decision = ?, decided_by = ?, decided_at = ? "
        "WHERE approval_id = ?",
        (decision, decided_by, time.strftime("%Y-%m-%dT%H:%M:%S"), approval_id),
    )
    conn.commit()
    conn.close()


def is_approved(approval_id):
    if FAULTS["skip_approval"]:
        return True                     # the fault: the gate always says yes
    conn = connect()
    row = conn.execute("SELECT decision FROM approvals WHERE approval_id = ?",
                       (approval_id,)).fetchone()
    conn.close()
    return row is not None and row["decision"] == "approved"


print("✅ Approval gate: request_approval / decide_approval / is_approved")

In [ ]:
# ============================================================================
# WRITE TOOLS: refuse to act without an approved approval row
# ============================================================================

@tool
def issue_refund(customer_id: str, invoice_id: str, amount: float,
                 approval_id: str = "") -> dict:
    """Refund an invoice. Requires an approved approval_id."""
    if not approval_id or not is_approved(approval_id):
        return {"status": "unavailable",
                "data": {"reason": "no approval", "approval_id": approval_id}}

    refund_id = "REF-" + hashlib.sha1(
        (invoice_id + str(amount)).encode()).hexdigest()[:8]
    conn = connect()
    # INSERT OR IGNORE: a retry must not produce a second refund. Notebook 06
    # checks "exactly once", and this is the line that makes it true.
    conn.execute(
        "INSERT OR IGNORE INTO refunds VALUES (?,?,?,?,?,?)",
        (refund_id, invoice_id, customer_id, amount, approval_id,
         time.strftime("%Y-%m-%dT%H:%M:%S")),
    )
    conn.execute("UPDATE invoices SET status = 'disputed' WHERE invoice_id = ?",
                 (invoice_id,))
    conn.commit()
    conn.close()
    return {"status": "ok", "data": {"refund_id": refund_id, "amount": amount}}


@tool
def change_plan(customer_id: str, new_plan: str, approval_id: str = "") -> dict:
    """Move a customer to a different plan. Requires an approved approval_id."""
    if not approval_id or not is_approved(approval_id):
        return {"status": "unavailable", "data": {"reason": "no approval"}}
    conn = connect()
    conn.execute("UPDATE accounts SET plan = ? WHERE customer_id = ?",
                 (new_plan, customer_id))
    conn.commit()
    conn.close()
    return {"status": "ok", "data": {"customer_id": customer_id, "plan": new_plan}}


@tool
def reset_password(customer_id: str) -> dict:
    """Send a password reset link. Safe: never returns or sets a password."""
    conn = connect()
    row = conn.execute("SELECT email FROM accounts WHERE customer_id = ?",
                       (customer_id,)).fetchone()
    conn.close()
    if row is None:
        return {"status": "empty", "data": None}
    return {"status": "ok", "data": {"sent_to": row["email"], "valid_minutes": 60}}


@tool
def create_ticket(customer_id: str, summary: str, priority: str = "normal") -> dict:
    """Open a support ticket for a human to pick up."""
    ticket_id = "TIC-" + hashlib.sha1(
        (customer_id + summary).encode()).hexdigest()[:8]
    conn = connect()
    conn.execute("INSERT OR IGNORE INTO tickets VALUES (?,?,?,?,?)",
                 (ticket_id, customer_id, summary, priority,
                  time.strftime("%Y-%m-%dT%H:%M:%S")))
    conn.commit()
    conn.close()
    return {"status": "ok", "data": {"ticket_id": ticket_id}}


TOOLS = {t.name: t for t in
         [get_account, get_invoice, issue_refund, change_plan,
          reset_password, create_ticket]}

# Which agent is allowed to use which tool. Notebook 07 asserts on this:
# "never call another agent's tools" is a policy, and policies need a table.
TOOL_OWNERSHIP = {
    "billing":   ["get_invoice", "issue_refund"],
    "account":   ["get_account", "change_plan", "reset_password"],
    "escalation": ["create_ticket"],
    "knowledge": [],
}

print("🔧 Tools registered:", ", ".join(sorted(TOOLS)))

---
## 📋 Part 4: The trajectory

One run in, one dict out. This is the most important data structure in the course — notebooks
05, 06, 07, 10, 11 and 12 all read it and nothing else.

```
trajectory = {
  "case_id", "input", "final_answer",
  "route",                     # which specialist the supervisor chose
  "steps": [                   # one per action, in order
     {"step", "agent", "tool", "args", "result_status", "handoff"}
  ],
  "tokens", "cost_usd",
  "final_state": {...}         # the DB, after
}
```

`result_status` is deliberately one of exactly three values — `ok`, `empty`, `unavailable` —
because "the tool worked", "the tool found nothing" and "the tool could not run" call for
three different agent behaviours, and notebook 05 scores whether the agent distinguished them.

In [ ]:
# ============================================================================
# TRAJECTORY RECORDER: append-only log of what the agent actually did
# ============================================================================
# Deliberately a plain dict, not a class. Every notebook downstream reads it as
# JSON, and a dict is the same thing in memory and on disk.

PRICE_PER_1K_TOKENS = 0.00060      # gpt-4o-mini blended; adjust for your model


def new_trajectory(case_id, user_input):
    return {
        "case_id": case_id,
        "input": user_input,
        "route": None,
        "steps": [],
        "tokens": 0,
        "cost_usd": 0.0,
        "final_answer": "",
        "final_state": {},
    }


def record_step(traj, agent, tool_name, args, result, handoff=None):
    """Append one action. `result` is a tool's {"status", "data"} dict."""
    traj["steps"].append({
        "step": len(traj["steps"]) + 1,
        "agent": agent,
        "tool": tool_name,
        "args": args,
        "result_status": result.get("status") if isinstance(result, dict) else "ok",
        "handoff": handoff,
    })


def record_tokens(traj, tokens):
    traj["tokens"] += tokens
    traj["cost_usd"] = round(traj["tokens"] / 1000.0 * PRICE_PER_1K_TOKENS, 6)


def snapshot_state():
    """The DB after the run. Notebook 06 diffs this against the expected state."""
    conn = connect()
    state = {
        "refunds": [dict(r) for r in conn.execute(
            "SELECT refund_id, invoice_id, customer_id, amount, approved_by "
            "FROM refunds ORDER BY refund_id").fetchall()],
        "tickets": [dict(r) for r in conn.execute(
            "SELECT ticket_id, customer_id, priority FROM tickets "
            "ORDER BY ticket_id").fetchall()],
        "plans": {r["customer_id"]: r["plan"] for r in conn.execute(
            "SELECT customer_id, plan FROM accounts").fetchall()},
        "approvals": [dict(r) for r in conn.execute(
            "SELECT approval_id, action, decision FROM approvals "
            "ORDER BY approval_id").fetchall()],
    }
    conn.close()
    return state


print("✅ Trajectory recorder ready")

---
## 🧭 Part 5: The supervisor and the specialists

The supervisor does one job — pick a route — and it is worth isolating precisely because it
is a *classification* problem hiding inside a generative system. Notebook 06 scores it with a
confusion matrix, which is only possible because the choice is a single labelled value rather
than a paragraph of reasoning.

In [ ]:
# ============================================================================
# SUPERVISOR: classify intent into exactly one route
# ============================================================================
ROUTES = ["knowledge", "billing", "account", "escalation"]

SUPERVISOR_PROMPT = """You route customer messages to one specialist.

knowledge  - questions about policy or documentation ("what is your refund window?")
billing    - invoices, charges, refunds, payment problems
account    - plans, passwords, team members, account settings
escalation - anger, legal threats, repeated failures, anything needing a human

Reply with exactly one word from: knowledge, billing, account, escalation.
If the message needs both documentation AND an action, choose the ACTION route -
the specialist can consult the knowledge agent itself."""


def supervise(user_input, traj):
    """Return one route string. Records its token cost on the trajectory."""
    system = SUPERVISOR_PROMPT
    if FAULTS["verbose_prompt"]:
        # The fault: someone pastes the whole style guide into the system prompt.
        # Nothing breaks. The bill triples.
        system = system + "\n\n" + ("Additional context. " * 400)

    text, tokens = call_model([
        {"role": "system", "content": system},
        {"role": "user", "content": user_input},
    ])
    record_tokens(traj, tokens)

    route = text.strip().lower().split()[0] if text.strip() else "escalation"
    if route not in ROUTES:
        route = "escalation"          # unparseable => a human looks at it

    if FAULTS["wrong_route"] and route == "billing":
        route = "account"             # the fault: billing work lands on account

    traj["route"] = route
    return route


print("🧭 Supervisor ready — routes:", ", ".join(ROUTES))

In [ ]:
# ============================================================================
# SPECIALIST AGENTS: each owns its tools and writes to the trajectory
# ============================================================================
# `handoff` is the payload one agent passes to the next. Keeping it explicit is
# what makes "did the context survive?" measurable in notebook 06.

def knowledge_agent(user_input, traj, handoff=None):
    """Answers from the documentation built in notebook 01."""
    # Notebook 01's index is re-opened here rather than imported: notebooks in
    # this course hand off through files, never through Python imports.
    import chromadb
    from sentence_transformers import SentenceTransformer

    client = chromadb.PersistentClient(path=os.path.join(DATA_DIR, "chroma"))
    coll = client.get_or_create_collection("lumen_kb")
    embedder = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    vector = [float(x) for x in embedder.encode([user_input],
                                                normalize_embeddings=True)[0]]
    hits = coll.query(query_embeddings=[vector], n_results=3)

    docs = hits["documents"][0] if hits["documents"] else []
    ids = [m["article_id"] for m in (hits["metadatas"][0] if hits["metadatas"] else [])]
    record_step(traj, "knowledge", "search_docs", {"query": user_input},
                {"status": "ok" if docs else "empty"}, handoff)

    context = "\n\n".join("[" + i + "] " + d for i, d in zip(ids, docs))
    text, tokens = call_model([
        {"role": "system",
         "content": "Answer only from the context, cite ids like [KB-001]. "
                    "If the context does not cover it, say you don't know.\n\n" + context},
        {"role": "user", "content": user_input},
    ])
    record_tokens(traj, tokens)
    return {"answer": text, "cited": ids, "findings": text[:300]}


def billing_agent(user_input, traj, handoff=None):
    """Invoices and refunds. The only agent that may call issue_refund."""
    customer_id = (handoff or {}).get("customer_id", "CUS-101")

    if FAULTS["lost_handoff_context"]:
        handoff = None                 # the fault: the policy finding never arrives

    tool_name = "get_invoice"
    args = {"customer_id": customer_id}
    if FAULTS["wrong_tool"]:
        tool_name, args = "get_account", {"customer_id": customer_id}
    if FAULTS["bad_arguments"]:
        args = {"customer_id": "CUS-999"}      # a customer that does not exist

    attempts = 3 if FAULTS["retry_loop"] else 1
    for _ in range(attempts):
        result = TOOLS[tool_name].invoke(args)
        record_step(traj, "billing", tool_name, args, result, handoff)

    invoices = result.get("data") or []
    if result.get("status") != "ok" or not invoices:
        # Honest handling of `empty`: say so, don't invent an invoice.
        return {"answer": "I couldn't find any invoices on that account.",
                "findings": "no invoices", "refund": None}

    # Find the duplicate: same amount, same day, different invoice id.
    by_key = defaultdict(list)
    for inv in invoices:
        by_key[(inv["amount"], inv["issued_on"])].append(inv)
    duplicates = [group for group in by_key.values() if len(group) > 1]

    if not duplicates:
        return {"answer": "I don't see a duplicate charge on this account.",
                "findings": "no duplicate", "refund": None}

    target = duplicates[0][0]
    approval_id = request_approval("issue_refund", {
        "customer_id": customer_id,
        "invoice_id": target["invoice_id"],
        "amount": target["amount"],
    })
    record_step(traj, "billing", "request_approval",
                {"action": "issue_refund", "invoice_id": target["invoice_id"]},
                {"status": "ok"}, handoff)

    return {"answer": "I found a duplicate charge of %.2f %s on %s and have requested a "
                      "refund, pending approval." % (target["amount"], target["currency"],
                                                     target["invoice_id"]),
            "findings": "duplicate " + target["invoice_id"],
            "refund": {"approval_id": approval_id, "customer_id": customer_id,
                       "invoice_id": target["invoice_id"], "amount": target["amount"]}}


def account_agent(user_input, traj, handoff=None):
    """Plans, passwords, settings."""
    customer_id = (handoff or {}).get("customer_id", "CUS-101")
    result = TOOLS["get_account"].invoke({"customer_id": customer_id})
    record_step(traj, "account", "get_account", {"customer_id": customer_id},
                result, handoff)
    data = result.get("data") or {}
    return {"answer": "Your account is on the %s plan in %s." % (
        data.get("plan", "unknown"), data.get("region", "unknown")),
        "findings": "plan " + str(data.get("plan"))}


def escalation_agent(user_input, traj, handoff=None):
    """Hands the conversation to a human."""
    customer_id = (handoff or {}).get("customer_id", "CUS-101")
    result = TOOLS["create_ticket"].invoke(
        {"customer_id": customer_id, "summary": user_input[:120], "priority": "high"})
    record_step(traj, "escalation", "create_ticket",
                {"customer_id": customer_id, "priority": "high"}, result, handoff)
    return {"answer": "I've opened a priority ticket and a specialist will reply shortly.",
            "findings": "escalated"}


AGENTS = {
    "knowledge": knowledge_agent,
    "billing": billing_agent,
    "account": account_agent,
    "escalation": escalation_agent,
}

print("👥 Agents ready:", ", ".join(sorted(AGENTS)))

---
## 🕸️ Part 6: The LangGraph assembly

The graph is thin on purpose — supervisor, specialist, approval, reply. Most of the
interesting behaviour lives in the agent functions, and the graph's job is to make the
*shape* of the run explicit so a trajectory metric has edges to compare against.

> **Note**: `langgraph` is pinned to `1.2.11` across this repository. Resolving `ragas` or
> `mlflow` can pull it back to `1.0.x`, which drops `langgraph.runtime.ExecutionInfo` and
> breaks the graph below. If that happens, re-pin it rather than floating it.

In [ ]:
# ============================================================================
# GRAPH STATE: framework-required TypedDict, kept deliberately flat
# ============================================================================
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class SupportState(TypedDict, total=False):
    case_id: str
    user_input: str
    customer_id: str
    route: str
    handoff: dict
    pending_refund: dict
    answer: str
    traj: dict


print("✅ State schema defined")

In [ ]:
# ============================================================================
# NODES: one function per box in the diagram
# ============================================================================

def node_supervisor(state):
    traj = state["traj"]
    route = supervise(state["user_input"], traj)
    return {"route": route,
            "handoff": {"customer_id": state.get("customer_id", "CUS-101"),
                        "issue": state["user_input"], "prior_findings": None}}


def node_knowledge(state):
    """Runs when the message needs policy AND an action: docs first, then billing."""
    traj = state["traj"]
    out = knowledge_agent(state["user_input"], traj, state.get("handoff"))
    handoff = dict(state.get("handoff") or {})
    handoff["prior_findings"] = out["findings"]      # what must survive the hop
    return {"handoff": handoff, "answer": out["answer"]}


def node_specialist(state):
    traj = state["traj"]
    agent = AGENTS[state["route"]]
    out = agent(state["user_input"], traj, state.get("handoff"))
    merged = (state.get("answer", "") + " " + out["answer"]).strip()
    return {"answer": merged, "pending_refund": out.get("refund")}


def node_approval(state):
    """The human decision. In production this is a queue; here it is one call."""
    pending = state.get("pending_refund")
    if not pending:
        return {}
    decide_approval(pending["approval_id"], "approved", "agent-reviewer@lumen")
    result = TOOLS["issue_refund"].invoke({
        "customer_id": pending["customer_id"],
        "invoice_id": pending["invoice_id"],
        "amount": pending["amount"],
        "approval_id": pending["approval_id"],
    })
    record_step(state["traj"], "billing", "issue_refund",
                {"invoice_id": pending["invoice_id"], "amount": pending["amount"]},
                result, state.get("handoff"))
    return {}


def route_after_supervisor(state):
    """Multi-intent messages visit the knowledge agent first."""
    text = state["user_input"].lower()
    wants_policy = any(w in text for w in ["policy", "policies", "política", "नीति", "rules"])
    if state["route"] in ("billing", "account") and wants_policy:
        return "knowledge"
    return "specialist"


def route_after_specialist(state):
    return "approval" if state.get("pending_refund") else END


print("✅ Nodes defined: supervisor, knowledge, specialist, approval")

In [ ]:
# ============================================================================
# GRAPH ASSEMBLY: wire the nodes, compile once
# ============================================================================
builder = StateGraph(SupportState)
builder.add_node("supervisor", node_supervisor)
builder.add_node("knowledge", node_knowledge)
builder.add_node("specialist", node_specialist)
builder.add_node("approval", node_approval)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_after_supervisor,
                              {"knowledge": "knowledge", "specialist": "specialist"})
builder.add_edge("knowledge", "specialist")
builder.add_conditional_edges("specialist", route_after_specialist,
                              {"approval": "approval", END: END})
builder.add_edge("approval", END)

graph = builder.compile()

print("🕸️  Graph compiled")
print("    START -> supervisor -> [knowledge] -> specialist -> [approval] -> END")

---
## 👁️ Part 7: Wrapping the run, with DeepEval spans

`@observe` from DeepEval creates a span per component. Notebook 06 uses those spans for
**blame attribution** — when an end-to-end run fails, which level failed first? Without spans
you get "the run was bad"; with them you get "the supervisor routed correctly, the tool call
was fine, and the reply dropped the citation".

In [ ]:
# ============================================================================
# RUN ONE CONVERSATION: graph + trajectory + DeepEval span
# ============================================================================
from deepeval.tracing import observe, update_current_span


@observe(type="agent")
def run_conversation(case_id, user_input, customer_id="CUS-101"):
    """Execute one case end to end. Returns the trajectory dict."""
    traj = new_trajectory(case_id, user_input)
    state = graph.invoke({
        "case_id": case_id,
        "user_input": user_input,
        "customer_id": customer_id,
        "traj": traj,
    })
    traj["final_answer"] = state.get("answer", "")
    traj["final_state"] = snapshot_state()

    update_current_span(input=user_input, output=traj["final_answer"])
    return traj


print("✅ run_conversation() ready — one call, one trajectory")

---
## ▶️ Part 8: See it work — the canonical run

*"I was double charged, and what's your refund policy?"* should produce: route `billing`, a
detour through `knowledge`, an invoice lookup, an approval request, a human decision, and
exactly one refund row.

In [ ]:
# ============================================================================
# SEE IT WORK: the canonical multi-step run
# ============================================================================
reset_database()

traj = run_conversation(
    "CASE-001",
    "I was double charged, and what's your refund policy?",
    customer_id="CUS-101",
)

print("🧭 route:", traj["route"])
print("🤖", traj["final_answer"][:280])
print()
print("📋 trajectory:")
for s in traj["steps"]:
    print("   %d. %-10s %-16s %-28s -> %s" % (
        s["step"], s["agent"], s["tool"], json.dumps(s["args"])[:28], s["result_status"]))
print()
print("💰 tokens: %d   cost: $%.5f" % (traj["tokens"], traj["cost_usd"]))
print("🗄️  refunds in DB:", traj["final_state"]["refunds"])

---
## 💥 Part 9: Break it

Three faults at three different levels. Watch what changes in the trajectory and what
doesn't — in every case the conversation still *completes* and still *reads fine*.

In [ ]:
# ============================================================================
# BREAK IT (1): wrong_tool and bad_arguments — the tool layer
# ============================================================================

def show(label, traj):
    print("--- %s ---" % label)
    for s in traj["steps"]:
        print("   %-10s %-16s %-30s -> %s" % (
            s["agent"], s["tool"], json.dumps(s["args"])[:30], s["result_status"]))
    print("   🤖", traj["final_answer"][:150])
    print("   🗄️  refunds:", len(traj["final_state"]["refunds"]))
    print()


reset_database()
FAULTS["wrong_tool"] = True
show("wrong_tool", run_conversation("CASE-001", "I was double charged.", "CUS-101"))
FAULTS["wrong_tool"] = False

reset_database()
FAULTS["bad_arguments"] = True
show("bad_arguments", run_conversation("CASE-001", "I was double charged.", "CUS-101"))
FAULTS["bad_arguments"] = False

print("⚠️  Both produce a polite, fluent, WRONG conversation and zero refunds.")
print("    Notebook 05 catches the first with ToolCorrectnessMetric and the second")
print("    with ArgumentCorrectnessMetric — they are genuinely different failures.")

In [ ]:
# ============================================================================
# BREAK IT (2): skip_approval — the safety fault
# ============================================================================
reset_database()
FAULTS["skip_approval"] = True

traj = run_conversation("CASE-001", "I was double charged.", "CUS-101")

conn = connect()
orphans = conn.execute(
    "SELECT r.refund_id, r.amount, a.decision FROM refunds r "
    "LEFT JOIN approvals a ON a.approval_id = r.approved_by").fetchall()
conn.close()

print("🗄️  refunds written:", [dict(r) for r in orphans])
print()
print("⚠️  The money moved. The trajectory looks IDENTICAL to a healthy run —")
print("    same tools, same order, same answer. Only the DB shows it.")
print("    That is why notebook 07's safety checks read state, not text,")
print("    and why a single violation blocks a release outright.")

FAULTS["skip_approval"] = False
reset_database()

In [ ]:
# ============================================================================
# BREAK IT (3): verbose_prompt and retry_loop — the cost faults
# ============================================================================
reset_database()
clean = run_conversation("CASE-001", "I was double charged.", "CUS-101")

reset_database()
FAULTS["verbose_prompt"] = True
FAULTS["retry_loop"] = True
costly = run_conversation("CASE-001", "I was double charged.", "CUS-101")
FAULTS["verbose_prompt"] = False
FAULTS["retry_loop"] = False

print("%-12s %8s %10s %6s" % ("run", "tokens", "cost", "steps"))
print("%-12s %8d %10.5f %6d" % ("clean", clean["tokens"], clean["cost_usd"],
                                len(clean["steps"])))
print("%-12s %8d %10.5f %6d" % ("faulty", costly["tokens"], costly["cost_usd"],
                                len(costly["steps"])))
print()
print("⚠️  Same answer, same refund, several times the price. No quality metric moves.")
print("    Notebook 11's fourth gate check is a cost ceiling for exactly this.")

---
## 🧪 Part 10: Try it yourself

**Exercise 1 — Make `empty` and `unavailable` behave differently.**
Turn on `tool_returns_empty` and read `billing_agent`. It currently says "I couldn't find any
invoices". Now imagine the tool had returned `unavailable` instead — a timeout. Should the
agent say the same sentence? Write the two-branch version. This distinction is the whole of
notebook 05's "result handling" section.

**Exercise 2 — Break the handoff and find it in the trajectory.**
Turn on `lost_handoff_context`, run the canonical case, and print `steps[i]["handoff"]` for
every step. The answer the customer sees barely changes. Write down, in one sentence, the
check you would need to catch it — then compare yours with notebook 06's handoff-integrity
metric.

---
## 📝 Recap and interview angle

### What this notebook built
- **A five-table SQLite database** that is the ground truth for every end-to-end check
- **Six tools**, with the two write tools gated on an `approvals` row
- **A supervisor plus four specialists**, assembled into a LangGraph
- **The trajectory dict** — the one structure the rest of the course reads
- **Record-and-replay fixtures**, so everything downstream is free and deterministic

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/lumen.db` | notebooks 03, 06, 07 (final-state checks) |
| `data/fixtures/` | every notebook — this is what makes replay possible |
| `data/sample_trajectory.json` | notebook 03 (shaping golden cases against a real run) |

### The interview angle
> *"How do you evaluate an agent, as opposed to a model?"*

The answer is the trajectory. A model produces one output you can score; an agent produces a
**sequence of decisions**, and the last one being fine does not mean the earlier ones were.
The three levels — tool call, agent/handoff, end-to-end — exist because a single end-to-end
score tells you that something is wrong and nothing about where.

The strongest thing you can say here is the `skip_approval` observation: a trajectory that is
byte-identical to a healthy one, with an unauthorised write sitting in the database. It makes
the case that **agent evaluation has to include state, not just text**, which is the part most
candidates miss and no popular framework does for you.

### Next
**[`03_the_golden_set.ipynb`](03_the_golden_set.ipynb)** — what a test case for an agent
actually contains, how to stratify one so small segments can't hide, and why at least a fifth
of the cases must have "refuse" as the correct answer.

In [ ]:
# ============================================================================
# SAVE ARTIFACTS: a clean database and one example trajectory
# ============================================================================
reset_database()
sample = run_conversation("CASE-001",
                          "I was double charged, and what's your refund policy?",
                          "CUS-101")

sample_path = os.path.join(DATA_DIR, "sample_trajectory.json")
with open(sample_path, "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)

reset_database()          # hand notebook 03 a clean slate

print("💾 Wrote", sample_path)
print("💾 Database reset at", DB_PATH)
print("🎬 Fixtures on disk:",
      len([f for f in os.listdir(FIXTURE_DIR) if f.endswith(".json")]))
print("✅ Notebook 02 complete — next: 03_the_golden_set.ipynb")